In [820]:
import numpy as np

# Load the test data

In [821]:
# noDataValue = 0
# bottom_value = np.float32(-10)
# array = np.genfromtxt("test_rasters/four_by_four.csv", delimiter=",")

noDataValue = -9999
bottom_value = np.float32(-100)
array = np.load("north-east_africa.npy")

# Calculate all the valid surface triangles

In [822]:
# A vertex in the array is valid if it's not equal to the noDataValue
valid_vertices = array != noDataValue

In [823]:
# Make 4 vertex arrays which tell whether the vertex for that cell is valid or not
top_left_vertices = valid_vertices[:-1, :-1]
bottom_left_vertices = valid_vertices[1:, :-1]
top_right_vertices = valid_vertices[:-1, 1:]
bottom_right_vertices = valid_vertices[1:, 1:]

In [824]:
top_left_triangles = top_left_vertices & bottom_left_vertices & top_right_vertices

In [825]:
bottom_right_triangles = bottom_left_vertices & top_right_vertices & bottom_right_vertices

In [826]:
is_orientation_2 = ~(top_left_triangles & bottom_right_triangles)

In [827]:
bottom_left_triangles = is_orientation_2 & (top_left_vertices & bottom_right_vertices & bottom_left_vertices)

In [828]:
top_right_triangles = is_orientation_2 & (top_left_vertices & bottom_right_vertices & top_right_vertices)

# Put the surface and floor vertices into an array

In [829]:
triangle_dtype = np.dtype([
    ("normal",  np.float32, (3,)),
    ("vertices", np.float32, (3,3,)),
    ("attr",    np.uint16),
], align=False)

In [830]:
def make_triangles(vertices):
    triangles = np.empty(len(vertices), dtype=triangle_dtype)
    triangles["vertices"] = vertices
    triangles["attr"] = 0
    return triangles

In [831]:
y, x = np.where(top_left_triangles)

bottom = np.full(len(y), bottom_value, dtype=np.float32)

top_left_portion_surface = make_triangles(np.stack([
                                        np.column_stack([x.astype(np.float32), y.astype(np.float32), array[y, x].astype(np.float32)]),
                                        np.column_stack([(x + 1).astype(np.float32), y.astype(np.float32), array[y, x + 1].astype(np.float32)]),
                                        np.column_stack([x.astype(np.float32), (y + 1).astype(np.float32), array[y + 1, x].astype(np.float32)]),
                                        ],
                                        axis=1))
top_left_portion_floor = make_triangles(np.stack([
                                        np.column_stack([x.astype(np.float32), y.astype(np.float32), bottom]),
                                        np.column_stack([x.astype(np.float32), (y + 1).astype(np.float32), bottom]), 
                                        np.column_stack([(x + 1).astype(np.float32), y.astype(np.float32), bottom]),
                                        ],
                                        axis=1))


In [832]:
y, x = np.where(bottom_right_triangles)

bottom = np.full(len(y), bottom_value, dtype=np.float32)

bottom_right_portion_surface = make_triangles(np.stack([
                                            np.column_stack([(x + 1).astype(np.float32), y.astype(np.float32), array[y, x + 1].astype(np.float32)]),
                                            np.column_stack([(x + 1).astype(np.float32), (y + 1).astype(np.float32), array[y + 1, x + 1].astype(np.float32)]),
                                            np.column_stack([x.astype(np.float32), (y + 1).astype(np.float32), array[y + 1, x].astype(np.float32)]),
                                            ],
                                            axis=1))
bottom_right_portion_floor = make_triangles(np.stack([
                                            np.column_stack([(x + 1).astype(np.float32), y.astype(np.float32), bottom]),
                                            np.column_stack([x.astype(np.float32), (y + 1).astype(np.float32), bottom]),
                                            np.column_stack([(x + 1).astype(np.float32), (y + 1).astype(np.float32), bottom]),
                                            ],
                                            axis=1))

In [833]:
y, x = np.where(bottom_left_triangles)

bottom = np.full(len(y), bottom_value, dtype=np.float32)

bottom_left_portion_surface = make_triangles(np.stack([
                                            np.column_stack([(x).astype(np.float32), (y).astype(np.float32), array[y, x].astype(np.float32)]),
                                            np.column_stack([(x + 1).astype(np.float32), (y + 1).astype(np.float32), array[y + 1, x + 1].astype(np.float32)]),
                                            np.column_stack([x.astype(np.float32), (y + 1).astype(np.float32), array[y + 1, x].astype(np.float32)]),
                                            ],
                                            axis=1))
bottom_left_portion_floor = make_triangles(np.stack([
                                            np.column_stack([(x).astype(np.float32), (y).astype(np.float32), bottom]),
                                            np.column_stack([x.astype(np.float32), (y + 1).astype(np.float32), bottom]),
                                            np.column_stack([(x + 1).astype(np.float32), (y + 1).astype(np.float32), bottom]),
                                            ],
                                            axis=1))

In [834]:
y, x = np.where(top_right_triangles)

bottom = np.full(len(y), bottom_value, dtype=np.float32)

top_right_portion_surface = make_triangles(np.stack([
                                        np.column_stack([(x + 1).astype(np.float32), y.astype(np.float32), array[y, x + 1].astype(np.float32)]),
                                        np.column_stack([(x + 1).astype(np.float32), (y + 1).astype(np.float32), array[y + 1, x + 1].astype(np.float32)]),
                                        np.column_stack([x.astype(np.float32), (y).astype(np.float32), array[y, x].astype(np.float32)]),
                                        ],
                                        axis=1))
top_right_portion_floor = make_triangles(np.stack([
                                        np.column_stack([(x + 1).astype(np.float32), y.astype(np.float32), bottom]),
                                        np.column_stack([x.astype(np.float32), (y).astype(np.float32), bottom]),
                                        np.column_stack([(x + 1).astype(np.float32), (y + 1).astype(np.float32), bottom]),
                                        ],
                                        axis=1))

# Put all the edge faces into an array

In [835]:
# Get all the triangle edges in the array
has_left_edge = (top_left_triangles | bottom_left_triangles)
has_right_edge = (top_right_triangles | bottom_right_triangles)
has_top_edge = (top_left_triangles | top_right_triangles)
has_bottom_edge = (bottom_left_triangles | bottom_right_triangles)

### Get all the triangle edges that aren't shared with another triangle

In [836]:
has_left_wall = np.copy(has_left_edge)
has_left_wall[:, 1:] = has_left_edge[:, 1:] & (~has_right_edge[:, :-1])

In [837]:
has_right_wall = np.copy(has_right_edge)
has_right_wall[:, :-1] = has_right_edge[:, :-1] & (~has_left_edge[:, 1:])

In [838]:
has_top_wall = np.copy(has_top_edge)
has_top_wall[1:, :] = has_top_edge[1:, :] & (~has_bottom_edge[:-1, :])

In [839]:
has_bottom_wall = np.copy(has_bottom_edge)
has_bottom_wall[:-1, :] = has_bottom_edge[:-1, :] & (~has_top_edge[1:, :])

In [840]:
has_up_diag_wall = top_left_triangles ^ bottom_right_triangles

In [841]:
has_down_diag_wall = bottom_left_triangles ^ top_right_triangles

### Add walls for the surface edges

In [842]:
y, x = np.where(has_left_wall)
bottom = np.full(len(y), bottom_value, dtype=np.float32)

left_wall_1 = make_triangles(np.stack([
                                        np.column_stack([(x).astype(np.float32), y.astype(np.float32), array[y, x].astype(np.float32)]),
                                        np.column_stack([(x).astype(np.float32), (y + 1).astype(np.float32), bottom]),
                                        np.column_stack([x.astype(np.float32), (y).astype(np.float32), bottom]),
                                        ],
                                        axis=1))
left_wall_2 = make_triangles(np.stack([
                                        np.column_stack([(x).astype(np.float32), y.astype(np.float32), array[y, x].astype(np.float32)]),
                                        np.column_stack([x.astype(np.float32), (y + 1).astype(np.float32), array[y + 1, x].astype(np.float32)]),
                                        np.column_stack([(x).astype(np.float32), (y + 1).astype(np.float32), bottom]),
                                        ],
                                        axis=1))

In [843]:
y, x = np.where(has_right_wall)
bottom = np.full(len(y), bottom_value, dtype=np.float32)

right_wall_1 = make_triangles(np.stack([
                                        np.column_stack([(x + 1).astype(np.float32), y.astype(np.float32), array[y, x + 1].astype(np.float32)]),
                                        np.column_stack([(x + 1).astype(np.float32), (y).astype(np.float32), bottom]),
                                        np.column_stack([(x + 1).astype(np.float32), (y + 1).astype(np.float32), bottom]),
                                        ],
                                        axis=1))
right_wall_2 = make_triangles(np.stack([
                                        np.column_stack([(x + 1).astype(np.float32), y.astype(np.float32), array[y, x + 1].astype(np.float32)]),
                                        np.column_stack([(x + 1).astype(np.float32), (y + 1).astype(np.float32), bottom]),
                                        np.column_stack([(x + 1).astype(np.float32), (y + 1).astype(np.float32), array[y + 1, x + 1].astype(np.float32)]),
                                        ],
                                        axis=1))

In [844]:
y, x = np.where(has_top_wall)
bottom = np.full(len(y), bottom_value, dtype=np.float32)

top_wall_1 = make_triangles(np.stack([
                                    np.column_stack([(x).astype(np.float32), y.astype(np.float32), array[y, x].astype(np.float32)]),
                                    np.column_stack([(x).astype(np.float32), (y).astype(np.float32), bottom]),
                                    np.column_stack([(x + 1).astype(np.float32), (y).astype(np.float32), bottom]),
                                ],
                                axis=1))
top_wall_2 = make_triangles(np.stack([
                                    np.column_stack([(x).astype(np.float32), y.astype(np.float32), array[y, x].astype(np.float32)]),
                                    np.column_stack([(x + 1).astype(np.float32), (y).astype(np.float32), bottom]),
                                    np.column_stack([(x + 1).astype(np.float32), (y).astype(np.float32), array[y, x + 1].astype(np.float32)]),
                                ],
                                axis=1))

In [845]:
y, x = np.where(has_bottom_wall)
bottom = np.full(len(y), bottom_value, dtype=np.float32)

bottom_wall_1 = make_triangles(np.stack([
                                        np.column_stack([(x).astype(np.float32), (y + 1).astype(np.float32), array[y + 1, x].astype(np.float32)]),
                                        np.column_stack([(x + 1).astype(np.float32), (y + 1).astype(np.float32), bottom]),
                                        np.column_stack([(x).astype(np.float32), (y + 1).astype(np.float32), bottom]),
                                    ],
                                    axis=1))
bottom_wall_2 = make_triangles(np.stack([
                                        np.column_stack([(x).astype(np.float32), (y + 1).astype(np.float32), array[y + 1, x].astype(np.float32)]),
                                        np.column_stack([(x + 1).astype(np.float32), (y + 1).astype(np.float32), array[y + 1, x + 1].astype(np.float32)]),
                                        np.column_stack([(x + 1).astype(np.float32), (y + 1).astype(np.float32), bottom]),
                                    ],
                                    axis=1))

In [846]:
y, x = np.where(has_up_diag_wall)
bottom = np.full(len(y), bottom_value, dtype=np.float32)

up_diag_wall_1 = make_triangles(np.stack([
                                        np.column_stack([(x + 1).astype(np.float32), y.astype(np.float32), array[y, x + 1].astype(np.float32)]),
                                        np.column_stack([x.astype(np.float32), (y + 1).astype(np.float32), array[y + 1, x].astype(np.float32)]),
                                        np.column_stack([x.astype(np.float32), (y + 1).astype(np.float32), bottom]),
                                        ],
                                        axis=1))
up_diag_wall_2 = make_triangles(np.stack([
                                        np.column_stack([(x + 1).astype(np.float32), y.astype(np.float32), array[y, x + 1].astype(np.float32)]),
                                        np.column_stack([x.astype(np.float32), (y + 1).astype(np.float32), bottom]),
                                        np.column_stack([(x + 1).astype(np.float32), (y).astype(np.float32), bottom]),
                                        ],
                                        axis=1))

In [847]:
y, x = np.where(has_down_diag_wall)
bottom = np.full(len(y), bottom_value, dtype=np.float32)

down_diag_wall_1 = make_triangles(np.stack([
                                    np.column_stack([(x).astype(np.float32), (y).astype(np.float32), array[y, x].astype(np.float32)]),
                                    np.column_stack([x.astype(np.float32), (y).astype(np.float32), bottom]),
                                    np.column_stack([(x + 1).astype(np.float32), (y + 1).astype(np.float32), bottom]),
                                    ],
                                    axis=1))
down_diag_wall_2 = make_triangles(np.stack([
                                    np.column_stack([(x).astype(np.float32), (y).astype(np.float32), array[y, x].astype(np.float32)]),
                                    np.column_stack([(x + 1).astype(np.float32), (y + 1).astype(np.float32), bottom]),
                                    np.column_stack([(x + 1).astype(np.float32), (y + 1).astype(np.float32), array[y + 1, x + 1].astype(np.float32)]),
                                    ],
                                    axis=1))

# Combine all the triangle arrays

In [848]:
triangles = np.concatenate([
    top_left_portion_surface, top_left_portion_floor,
    bottom_right_portion_surface, bottom_right_portion_floor,
    bottom_left_portion_surface, bottom_left_portion_floor,
    top_right_portion_surface, top_right_portion_floor,
    left_wall_1, left_wall_2,
    right_wall_1, right_wall_2,
    top_wall_1, top_wall_2,
    bottom_wall_1, bottom_wall_2,
    up_diag_wall_1, up_diag_wall_2,
    down_diag_wall_1, down_diag_wall_2,
])

# Write the STL file

In [849]:
# Calculate normals
v0 = triangles["vertices"][:, 0]
v1 = triangles["vertices"][:, 1]
v2 = triangles["vertices"][:, 2]

edge1 = v1 - v0
edge2 = v2 - v0

normals = np.cross(edge1, edge2)

lengths = np.linalg.norm(normals, axis=1)

valid = lengths > 0
normals[valid] /= lengths[valid, None]

triangles["normal"] = normals

In [850]:
with open("mesh.stl", "wb") as f:
    # Write the header of the binary STL
    f.write(b"\0" * 80)

    # Write in the number of triangles
    f.write(np.uint32(len(triangles)).tobytes())
    
    # Write in the surface faces
    f.write(triangles.tobytes())